In [0]:
%run /ImaginaryCompany/ImaginaryWork/includes/ImaginaryCompanyImaginaryWorkCore 

# facttest (262765)
## Description
Fact table defining per curation metrics around dumb meter reads.

## Source Tables

|sourceTablEntityId|sourceTable|source Table Description|Further Information - Primary Keys, Reasons for Filters, Business Context|
|------||-----------|------------------------|-------------------|



## Lineage
Link to lineage diagram on ImaginaryCompany Hub https://ImaginaryCompanyhub.azurewebsites.net/relationship-dependency

(update to your entity, add this link to Data Catalog too)

## Limitations


## Assumptions


## History
1. First built by Squad 2 - https://dev.azure.com/Imaginarywater/Data%20Services/_sprints/taskboard/Squad%202/Data%20Services/Development/Squad%202/dumb%20Metering%20Phase%202%20Design%20%5BSprint%2049%5D?workitem=85400

In [0]:
from ImaginaryCompany.ImaginaryWork.core import classTest, classFactTest, classTestLog, classTestEnTEx, classTestDel, classTestPro
import ImaginaryCompany.ImaginaryWork.core.functionDataframe as fd
import ImaginaryCompany.ImaginaryWork.function.test.functionTest as ft
import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta
env = classTestPro().env

In [0]:
dbutils.widgets.text("batchId", "")
dbutils.widgets.text("entityId", "262765")

In [0]:
### Get widget parameters
try:
  dbutils.widgets.get("batchId")
  batchId = getArgument("batchId")
except:
  pass
dbutils.widgets.get("entityId")
entityId = getArgument("entityId")
if not entityId:
  dbutils.notebook.exit("Cannot continue processing without an entityId")

In [0]:
## Register the start of the processing
batchEntity = classTest(batchId, entityId)
batchEntity.start(entityId)
## Get the batch Id
batchId = batchEntity.getId()
## Get an instance of classFactTest and start the preProcess
fact = classFactTest(entityId)
fact.preProcess()


In [0]:
# we only want  readings
otReads = spark.sql(f"SELECT * FROM otdumbMeterReadsLatestFull WHERE MINUTE(recordTime) = 0")
otReads.createOrReplaceTempView('otdumbMeterReadsLatestFull')

In [0]:
curatedExists = fact.targetExists
deltaDate =  str(fact.deltaDate)

In [0]:
meterStatus = spark.sql(f"""
select 
*, 
CAST(ImaginaryCompany_{env}.function.dateToSid(cast(ImaginaryCompanyMetadataValidFrom as TIMESTAMP) ) as INT) as validFromSid,
CAST(ImaginaryCompany_{env}.function.dateToSid(cast(ImaginaryCompanyMetadataValidTo as TIMESTAMP) ) as INT) as validToSid
from dimdumbMeterMeterStatusFull""")
meterStatus.createOrReplaceTempView('''meterStatus''')

In [0]:
# we shift a day backwards to te that prompted the operats change
operationalMeterStatus = spark.sql(f"""
select 
*,
CAST(ImaginaryCompany_{env}.function.dateToSid(date_add(cast(ImaginaryCompanyMetadataValidFrom as TIMESTAMP),-1) ) as INT) as validFromSid,
CAST(ImaginaryCompany_{env}.function.dateToSid(date_add(cast(ImaginaryCompanyMetadataValidTo as TIMESTAMP),-1) ) as INT) as validToSid
from dimdumbMeterOperationalMeterStatusFull """)
operationalMeterStatus.createOrReplaceTempView('''operationalMeterStatus''')

In [0]:
if curatedExists is False:
  
  # the earliest load time in the ot file
  minDate = str(spark.sql("""
  select min(ImaginaryCompanyMetadataLoadDateTime) from otdumbMeterReadsLatestFull
  """).collect()[0][0])[:10]
  minDateSid = int(minDate.replace('-',''))
  maxDate = str(datetime.date.today())[:10]
  maxDateSid = int(maxDate.replace('-',''))
  
  # query to gather no of reads per amrid, per record date and per curation date
  otDedup = spark.sql(f"""

  --de-duplicate otDelta table
  SELECT 
    mm.amrid as dumbMeterCurationDailyAmrid,
    CAST(ImaginaryCompany_{env}.function.dateToSid(cast(mm.RecordTime as TIMESTAMP) ) as INT)                as dumbMeterCurationDailyRecordDateSid,
    CAST(ImaginaryCompany_{env}.function.dateToSid(cast(mm.ReceivedDateTime as TIMESTAMP) ) as INT)          as dumbMeterCurationDailyReceivedDateSid,
    mm.ReceivedDateTime                                                      as dumbMeterCurationDailyReceivedDateTime,
    CAST(ImaginaryCompany_{env}.function.dateToSid(cast(mm.ImaginaryCompanyMetadataLoadDateTime as TIMESTAMP) ) as INT)  as dumbMeterCurationDailyCurationDateSid,
    mm.ImaginaryCompanyMetadataLoadDateTime                                              as dumbMeterCurationDailyCurationDateTime,
    CASE WHEN HOUR(mm.RecordTime) = 23 THEN 1 ELSE 0 END                     as dumbMeterCurationDailyHas11pmReadInd,
    CASE WHEN HOUR(mm.RecordTime) = 0 THEN 1 ELSE 0 END                      as dumbMeterCurationDailyHasMidnightReadInd
  FROM 
    (
      SELECT
        amrid,
        recordTime,
        receivedDateTime,
        ImaginaryCompanyMetadataLoadDateTime,
        meterReading,
        row_number() OVER(PARTITION BY amrid,recordTime ORDER BY id) AS rowNumber
      FROM
        otdumbMeterReadsLatestFull
    ) mm

  WHERE
    mm.rowNumber = 1 
  """)
  otDedup.createOrReplaceTempView('otDedup')



  otDailyReads = spark.sql("""
  SELECT
    dumbMeterCurationDailyAmrid,
    dumbMeterCurationDailyRecordDateSid,
    dumbMeterCurationDailyReceivedDateSid,
    dumbMeterCurationDailyReceivedDateTime,
    dumbMeterCurationDailyCurationDateSid,
    dumbMeterCurationDailyCurationDateTime,
    COUNT(*)                                              as dumbMeterCurationDailyNoOfReadsInCuration,
    MAX(dumbMeterCurationDailyHas11pmReadInd)            as dumbMeterCurationDailyHas11pmReadInd,
    MAX(dumbMeterCurationDailyHasMidnightReadInd)        as dumbMeterCurationDailyHasMidnightReadInd,
    DENSE_RANK () OVER ( 
            PARTITION BY dumbMeterCurationDailyCurationDateSid
            ORDER BY dumbMeterCurationDailyCurationDateTime ASC
        ) dumbMeterCurationDailyCurationNumber
  FROM otDedup
  GROUP BY 
    dumbMeterCurationDailyAmrid, 
    dumbMeterCurationDailyRecordDateSid, 
    dumbMeterCurationDailyReceivedDateSid,
    dumbMeterCurationDailyReceivedDateTime,
    dumbMeterCurationDailyCurationDateSid,
    dumbMeterCurationDailyCurationDateTime
  """)
  otDailyReads.createOrReplaceTempView('otDailyReads')
  
  # Query to add relevent sids to dimdumbMeter
  # this enables us to join this fact on sids to relevant dims (dimAddress, dimPremise, dimUnIns).
  dsm = spark.sql("""
  SELECT 
  dsm.dumbMeterSid as dumbMeterSid,
  dsm.dumbMeterAmrid as dumbMeterAmrid,
  rel.dumbMeterRelationshipStrictUnInsSid as dumbMeterUnInsSid,
  rel.premiseSid as dumbMeterPremiseSid,
  da.addressSid as dumbMeterAddressSid
  FROM dimdumbMeterFull dsm
  LEFT JOIN (
    select distinct dumbMeterRelationshipStrictMeterAmrid,
      dumbMeterRelationshipStrictMeterSerialNumber,
      dumbMeterRelationshipStrictEquipmentId,
      dumbMeterRelationshipStrictUnInsSid,
      dumbMeterRelationshipStrictMeterAddressNumber,
      dp.premiseSid as premiseSid
    FROM datasetdumbMeterRelationshipStrictFull
    LEFT JOIN  dimPremiseFull dp 
    ON dp.premiseNumber = datasetdumbMeterRelationshipStrictFull.dumbMeterRelationshipStrictUnInsPremiseNumber
    WHERE dumbMeterRelationshipStrictValidTo >'9998-12-31T23:00:00.000+0000'
    AND dumbMeterRelationshipStrictValidInd = 1
  ) rel 
  ON dsm.dumbMeterAmrid = rel.dumbMeterRelationshipStrictMeterAmrid
  AND dsm.dumbMeterMeterSerialNumber = rel.dumbMeterRelationshipStrictMeterSerialNumber
  AND dsm.dumbMeterEquipmentId = rel.dumbMeterRelationshipStrictEquipmentId
  LEFT JOIN dimAddressFull da
  ON rel.dumbMeterRelationshipStrictMeterAddressNumber = cast(da.addressId as int)
  --WHERE dsm.dumbMeterSapRecordInd = 1
  """)
  dsm.createOrReplaceTempView('dsm')
  
  # This query generates the spine of the fact
  # we must have at least 1 row for each dumb meter per day.
  # In the case that no dumb meter data exists for a certain day -
  # using this cross table we will be able to report reads for that day as 0.
  crossTable = spark.sql("""
  SELECT      
    dd.dateSid as dateSid,
    dd.dateCalendarYearWeek as dateCalendarYearWeek,
    dsm.dumbMeterAmrid as dumbMeterAmrid,
    dsm.dumbMeterSid as dumbMeterCurationDailydumbMeterSid,
    dsm.dumbMeterUnInsSid as dumbMeterCurationDailyUnInsSid,
    dsm.dumbMeterAddressSid as dumbMeterCurationDailyAddressSid,
    dsm.dumbMeterPremiseSid as dumbMeterCurationDailyPremiseSid

  FROM        dimDateFull dd
  CROSS JOIN  dsm
  WHERE       dd.dateSid>={0}
  AND         dd.datesid<{1}

  """.format(minDateSid,maxDateSid))
  crossTable.createOrReplaceTempView('crossTable')
  
  # We join our dumb meter reads to the cross table using record date
  # this means for each dumb meter, we have a row for every record date.
  # We also join the meter status tables on their relevant valid from and to dates.
  baseJoined = spark.sql("""
  SELECT 
    bt.dateSid as dumbMeterCurationDailyDateSid,
    bt.dateCalendarYearWeek as dumbMeterCurationDailyDateCalendarYearWeek,
    bt.dumbMeterCurationDailydumbMeterSid as dumbMeterCurationDailydumbMeterSid,
    bt.dumbMeterCurationDailyUnInsSid as dumbMeterCurationDailyUnInsSid,
    bt.dumbMeterCurationDailyAddressSid as dumbMeterCurationDailyAddressSid,
    bt.dumbMeterCurationDailyPremiseSid as dumbMeterCurationDailyPremiseSid,
    NVL(dsmms.dumbMeterMeterStatusSid,-1) as dumbMeterCurationDailyMeterStatusSid,
    NVL(dsmoms.dumbMeterOperationalMeterStatusSid,-1) as dumbMeterCurationDailyOperationalMeterStatusSid,
    NVL(ot.dumbMeterCurationDailyRecordDateSid, bt.dateSid) as dumbMeterCurationDailyRecordDateSid,
    NVL(ot.dumbMeterCurationDailyReceivedDateSid, -1) as dumbMeterCurationDailyReceivedDateSid,
    NVL(ot.dumbMeterCurationDailyReceivedDateTime, '1900-01-01 00:00:00') as dumbMeterCurationDailyReceivedDateTime,
    NVL(ot.dumbMeterCurationDailyCurationDateSid, -1) as dumbMeterCurationDailyCurationDateSid,
    NVL(ot.dumbMeterCurationDailyCurationDateTime, '1900-01-01 00:00:00') as dumbMeterCurationDailyCurationDateTime,
    NVL(ot.dumbMeterCurationDailyNoOfReadsInCuration, 0) as dumbMeterCurationDailyNoOfReadsInCuration,
    NVL(ot.dumbMeterCurationDailyHas11pmReadInd, 0) as dumbMeterCurationDailyHas11pmReadInd,
    NVL(ot.dumbMeterCurationDailyHasMidnightReadInd, 0) as dumbMeterCurationDailyHasMidnightReadInd,
    NVL(ot.dumbMeterCurationDailyCurationNumber, 0)  as dumbMeterCurationDailyCurationNumber

  FROM crossTable bt
  LEFT JOIN otDailyReads ot on ot.dumbMeterCurationDailyAmrid = bt.dumbMeterAmrid AND ot.dumbMeterCurationDailyRecordDateSid = bt.dateSid 
  LEFT JOIN meterStatus dsmms on dsmms.dumbMeterMeterStatusAmrid = bt.dumbMeterAmrid and dsmms.validFromSid <= bt.dateSid and dsmms.validToSid > bt.dateSid
  LEFT JOIN operationalMeterStatus dsmoms on dsmoms.dumbMeterOperationalMeterStatusAmrid = bt.dumbMeterAmrid and dsmoms.validFromSid <= bt.dateSid and dsmoms.validToSid > bt.dateSid
  """)

else:
  # filtering dumb meter reads file for past 7 days to improve performance
  deltaDateTime = datetime.datetime.now() - relativedelta(days=7)
  delta = classTestDel('otdumbMeterReadsLatest', 
                     #location = '/mnt/dataproc/base/ot/otdumbMeterReads/otdumbMeterReadsLatest', 
                     transactionType = 'latest', partitionBy = 'ImaginaryCompanyMetadataLoadYear,ImaginaryCompanyMetadataLoadMonth,ImaginaryCompanyMetadataLoadDay')
  partitionFilter = delta.buildMetadataDateBasedPredicates(deltaDateTime)
  print(partitionFilter)
  otReads = spark.sql(f"SELECT * FROM otdumbMeterReadsLatestFull WHERE {partitionFilter}")
  otReads.createOrReplaceTempView('otReadsFromUtc')
  
  deltaDateTime = datetime.datetime.fromisoformat(deltaDate) + timedelta(minutes=1)
  todayDate = str(datetime.datetime.now().date())
  print(deltaDateTime, todayDate)
  
  # Gathering all the relevant dumb meter data based on the most recent run (deltaDateTime) and today's date.
  
  # We must ignore all records from today as the meter status wont be correct until the day after, 
  # so we must not append these rows to the fact until the next day.
  # However we must include the reads from the full day of the last deltaDate, as well as everything from that deltaDate up to yesterday.
  otDelta = spark.sql(f"""
  SELECT
    *
  FROM
    otReadsFromUtc
  WHERE
    CASE
      WHEN CAST(recordTime AS date) = CAST('{deltaDateTime}' AS date) 
        THEN CAST(recordTime as date) <= CAST('{todayDate}' AS date) - 1 AND CAST(ImaginaryCompanyMetaDataLoadDateTime as date) >= cast('{deltaDateTime}' AS date)
      WHEN CAST('{todayDate}' AS date) = CAST('{deltaDateTime}' AS date) 
        THEN CAST(recordTime AS date) <= CAST('{todayDate}' as date) - 1 AND ImaginaryCompanyMetaDataLoadDateTime > '{deltaDateTime}'
      ELSE CAST(recordTime AS date) <= CAST('{todayDate}' as date) - 1 AND CAST(ImaginaryCompanyMetaDataLoadDateTime AS date) > CAST('{deltaDateTime}' AS date)
    END
  """)
  otDelta.createOrReplaceTempView("""otDelta""")

  
  minDate = str(spark.sql("""
  select min(ImaginaryCompanyMetadataLoadDateTime) from otDelta
  """).collect()[0][0])[:10]
  if minDate == 'None':
    minDateSid = 99991231
  else:
    minDateSid = int(minDate.replace('-',''))
  maxDate = str(datetime.date.today())[:10]
  maxDateSid = int(maxDate.replace('-',''))
  deltaDay = str(deltaDate)[:10]
  if deltaDay == 'None':
    deltaDaySid = maxDateSid
  else:
    deltaDaySid = int(deltaDay.replace('-',''))
  
  
  # query to gather no of reads per amrid, per record date and per curation date
  otDedup = spark.sql(f"""

  --de-duplicate otDelta table
  SELECT 
    mm.amrid as dumbMeterCurationDailyAmrid,
    CAST(ImaginaryCompany_{env}.function.dateToSid(cast(mm.RecordTime as TIMESTAMP) ) as INT)                as dumbMeterCurationDailyRecordDateSid,
    CAST(ImaginaryCompany_{env}.function.dateToSid(cast(mm.ReceivedDateTime as TIMESTAMP) ) as INT)          as dumbMeterCurationDailyReceivedDateSid,
    mm.ReceivedDateTime                                                      as dumbMeterCurationDailyReceivedDateTime,
    CAST(ImaginaryCompany_{env}.function.dateToSid(cast(mm.ImaginaryCompanyMetadataLoadDateTime as TIMESTAMP) ) as INT)  as dumbMeterCurationDailyCurationDateSid,
    mm.ImaginaryCompanyMetadataLoadDateTime                                              as dumbMeterCurationDailyCurationDateTime,
    CASE WHEN HOUR(mm.RecordTime) = 23 THEN 1 ELSE 0 END                     as dumbMeterCurationDailyHas11pmReadInd,
    CASE WHEN HOUR(mm.RecordTime) = 0 THEN 1 ELSE 0 END                      as dumbMeterCurationDailyHasMidnightReadInd
  FROM 
    (
      SELECT
        amrid,
        recordTime,
        receivedDateTime,
        ImaginaryCompanyMetadataLoadDateTime,
        meterReading,
        row_number() OVER(PARTITION BY amrid,recordTime ORDER BY id) AS rowNumber
      FROM
        otDelta
    ) mm

  WHERE
    mm.rowNumber = 1 
  
  -- join all distinct curation times in order to calculate accurate curation number
  UNION ALL
  
  SELECT DISTINCT
  1 as dumbMeterCurationDailyAmrid,
  19000000 as dumbMeterCurationDailyRecordDateSid,
  19000000 as dumbMeterCurationDailyReceivedDateSid,
  '1900-01-01 00:00:00' as dumbMeterCurationDailyReceivedDateTime,
  CAST(ImaginaryCompany_{env}.function.dateToSid(cast(ImaginaryCompanyMetadataLoadDateTime as TIMESTAMP) ) as INT)  as dumbMeterCurationDailyCurationDateSid,
  ImaginaryCompanyMetadataLoadDateTime as dumbMeterCurationDailyCurationDateTime,
  -1 as dumbMeterCurationDailyHas11pmReadInd,
  -1 as dumbMeterCurationDailyHasMidnightReadInd
  from otdumbMeterReadsLatestFull
  """)
  otDedup.createOrReplaceTempView('otDedup')



  otDailyReads = spark.sql("""
  SELECT
    dumbMeterCurationDailyAmrid,
    dumbMeterCurationDailyRecordDateSid,
    dumbMeterCurationDailyReceivedDateSid,
    dumbMeterCurationDailyReceivedDateTime,
    dumbMeterCurationDailyCurationDateSid,
    dumbMeterCurationDailyCurationDateTime,
    COUNT(*)                                              as dumbMeterCurationDailyNoOfReadsInCuration,
    MAX(dumbMeterCurationDailyHas11pmReadInd)            as dumbMeterCurationDailyHas11pmReadInd,
    MAX(dumbMeterCurationDailyHasMidnightReadInd)        as dumbMeterCurationDailyHasMidnightReadInd,
    DENSE_RANK () OVER ( 
            PARTITION BY dumbMeterCurationDailyCurationDateSid
            ORDER BY dumbMeterCurationDailyCurationDateTime ASC
        ) dumbMeterCurationDailyCurationNumber
  FROM otDedup
  GROUP BY 
    dumbMeterCurationDailyAmrid, 
    dumbMeterCurationDailyRecordDateSid, 
    dumbMeterCurationDailyReceivedDateSid, 
    dumbMeterCurationDailyReceivedDateTime,
    dumbMeterCurationDailyCurationDateSid,
    dumbMeterCurationDailyCurationDateTime

  """)
  otDailyReads.createOrReplaceTempView('otDailyReads')
  
  # remove dummy rows used to calculate curation number
  otDailyReads = spark.sql("""
  SELECT * from otDailyReads
  WHERE dumbMeterCurationDailyHasMidnightReadInd  <> -1
  """)
  otDailyReads.createOrReplaceTempView('otDailyReads')
  
  # Query to add relevent sids to dimdumbMeter
  # this enables us to join this fact on sids to relevant dims (dimAddress, dimPremise, dimUnIns).
  dsm = spark.sql("""
  SELECT 
    dsm.dumbMeterSid as dumbMeterSid,
    dsm.dumbMeterAmrid as dumbMeterAmrid,
    rel.dumbMeterRelationshipStrictUnInsSid as dumbMeterUnInsSid,
    rel.premiseSid as dumbMeterPremiseSid,
    da.addressSid as dumbMeterAddressSid
  FROM dimdumbMeterFull dsm
  LEFT JOIN (
    SELECT DISTINCT dumbMeterRelationshipStrictMeterAmrid,
      dumbMeterRelationshipStrictMeterSerialNumber,
      dumbMeterRelationshipStrictEquipmentId,
      dumbMeterRelationshipStrictUnInsSid,
      dumbMeterRelationshipStrictMeterAddressNumber,
      dp.premiseSid as premiseSid
    FROM datasetdumbMeterRelationshipStrictFull
    LEFT JOIN  dimPremiseFull dp 
    ON dp.premiseNumber = datasetdumbMeterRelationshipStrictFull.dumbMeterRelationshipStrictUnInsPremiseNumber
    WHERE dumbMeterRelationshipStrictValidTo >'9998-12-31T23:00:00.000+0000'
    AND dumbMeterRelationshipStrictValidInd = 1

    ) rel 
  ON dsm.dumbMeterAmrid = rel.dumbMeterRelationshipStrictMeterAmrid
  AND dsm.dumbMeterMeterSerialNumber = rel.dumbMeterRelationshipStrictMeterSerialNumber
  AND dsm.dumbMeterEquipmentId = rel.dumbMeterRelationshipStrictEquipmentId
  LEFT JOIN dimAddressFull da
  ON rel.dumbMeterRelationshipStrictMeterAddressNumber = cast(da.addressId as int)
  --WHERE dsm.dumbMeterSapRecordInd = 1
  """)
  dsm.createOrReplaceTempView('dsm')

  
  # This query extends the spine of the fact
  # we must have at least 1 row for each dumb meter per day.
  # In the case that no dumb meter data exists for a certain day -
  # using this cross table we will be able to report reads for that day as 0.
  
  # As this is a delta run we only need to generate a row per meter from the last deltaDate up until yesterday.
  # We don't include the current date as we will cannot append any rows until the day is complete.
  crossTable1 = spark.sql("""
  SELECT      
    dd.dateSid as dateSid,
    dd.dateCalendarYearWeek as dateCalendarYearWeek,
    dsm.dumbMeterAmrid as dumbMeterAmrid,
    dsm.dumbMeterSid as dumbMeterCurationDailydumbMeterSid,
    dsm.dumbMeterUnInsSid as dumbMeterCurationDailyUnInsSid,
    dsm.dumbMeterAddressSid as dumbMeterCurationDailyAddressSid,
    dsm.dumbMeterPremiseSid as dumbMeterCurationDailyPremiseSid

  FROM        dimDateFull dd
  CROSS JOIN  dsm
  WHERE       dd.dateSid<{0}
  and         dd.dateSid>={1}

  """.format(maxDateSid, deltaDaySid))
  crossTable1.createOrReplaceTempView('crossTable1')
  
  # In the delta run we will also recieve reads that are outside of the date range in our first crossTable1 above.
  # We must generate rows to cater for the extra reads outside of the already defined date range.
  crossTable2 = spark.sql("""
  SELECT   
    ot.dumbMeterCurationDailyRecordDateSid as dateSid,
    dd.dateCalendarYearWeek as dateCalendarYearWeek,
    dsm.dumbMeterAmrid as dumbMeterAmrid,
    dsm.dumbMeterSid as dumbMeterCurationDailydumbMeterSid,
    dsm.dumbMeterUnInsSid as dumbMeterCurationDailyUnInsSid,
    dsm.dumbMeterAddressSid as dumbMeterCurationDailyAddressSid,
    dsm.dumbMeterPremiseSid as dumbMeterCurationDailyPremiseSid

  FROM        otDailyReads ot
  LEFT JOIN   dimDateFull dd on dd.dateSid = ot.dumbMeterCurationDailyRecordDateSid
  INNER JOIN  dsm on dsm.dumbMeterAmrid = ot.dumbMeterCurationDailyAmrid
  WHERE       ot.dumbMeterCurationDailyRecordDateSid<{0}
  """.format(maxDateSid))
  crossTable2.createOrReplaceTempView('crossTable2')
  
  # We join our dumb meter reads to crossTable1 using record date
  # this means for each dumb meter, we have a row for every record date.
  # We also join the meter status tables on their relevant valid from and to dates.
  todayReads = spark.sql("""
  SELECT 
    bt.dateSid as dumbMeterCurationDailyDateSid,
    bt.dateCalendarYearWeek as dumbMeterCurationDailyDateCalendarYearWeek,
    bt.dumbMeterCurationDailydumbMeterSid as dumbMeterCurationDailydumbMeterSid,
    bt.dumbMeterCurationDailyUnInsSid as dumbMeterCurationDailyUnInsSid,
    bt.dumbMeterCurationDailyAddressSid as dumbMeterCurationDailyAddressSid,
    bt.dumbMeterCurationDailyPremiseSid as dumbMeterCurationDailyPremiseSid,
    NVL(dsmms.dumbMeterMeterStatusSid,-1) as dumbMeterCurationDailyMeterStatusSid,
    NVL(dsmoms.dumbMeterOperationalMeterStatusSid,-1) as dumbMeterCurationDailyOperationalMeterStatusSid,
    NVL(ot.dumbMeterCurationDailyRecordDateSid, bt.dateSid) as dumbMeterCurationDailyRecordDateSid,
    NVL(ot.dumbMeterCurationDailyReceivedDateSid, -1) as dumbMeterCurationDailyReceivedDateSid,
    NVL(ot.dumbMeterCurationDailyReceivedDateTime, '1900-01-01 00:00:00') as dumbMeterCurationDailyReceivedDateTime,
    NVL(ot.dumbMeterCurationDailyCurationDateSid, -1) as dumbMeterCurationDailyCurationDateSid,
    NVL(ot.dumbMeterCurationDailyCurationDateTime, '1900-01-01 00:00:00') as dumbMeterCurationDailyCurationDateTime,
    NVL(ot.dumbMeterCurationDailyNoOfReadsInCuration, 0) as dumbMeterCurationDailyNoOfReadsInCuration,
    NVL(ot.dumbMeterCurationDailyHas11pmReadInd, 0) as dumbMeterCurationDailyHas11pmReadInd,
    NVL(ot.dumbMeterCurationDailyHasMidnightReadInd, 0) as dumbMeterCurationDailyHasMidnightReadInd,
    NVL(ot.dumbMeterCurationDailyCurationNumber, 0)  as dumbMeterCurationDailyCurationNumber

  FROM crossTable1 bt
  LEFT JOIN otDailyReads ot on ot.dumbMeterCurationDailyAmrid = bt.dumbMeterAmrid AND ot.dumbMeterCurationDailyRecordDateSid = bt.dateSid 
  LEFT JOIN meterStatus dsmms on dsmms.dumbMeterMeterStatusAmrid = bt.dumbMeterAmrid and dsmms.validFromSid <= bt.dateSid and dsmms.validToSid > bt.dateSid
  LEFT JOIN operationalMeterStatus dsmoms on dsmoms.dumbMeterOperationalMeterStatusAmrid = bt.dumbMeterAmrid and dsmoms.validFromSid <= bt.dateSid and dsmoms.validToSid > bt.dateSid
  """)
  todayReads.createOrReplaceTempView("""todayReads""")
  
  # We also join our dumb meter reads to the crossTable2 using record date
  # this means for each dumb meter, we have a row for every record date.
  # We also join the meter status tables on their relevant valid from and to dates.
  previousReads = spark.sql("""
  SELECT 
    bt.dateSid as dumbMeterCurationDailyDateSid,
    bt.dateCalendarYearWeek as dumbMeterCurationDailyDateCalendarYearWeek,
    bt.dumbMeterCurationDailydumbMeterSid as dumbMeterCurationDailydumbMeterSid,
    bt.dumbMeterCurationDailyUnInsSid as dumbMeterCurationDailyUnInsSid,
    bt.dumbMeterCurationDailyAddressSid as dumbMeterCurationDailyAddressSid,
    bt.dumbMeterCurationDailyPremiseSid as dumbMeterCurationDailyPremiseSid,
    NVL(dsmms.dumbMeterMeterStatusSid,-1) as dumbMeterCurationDailyMeterStatusSid,
    NVL(dsmoms.dumbMeterOperationalMeterStatusSid,-1) as dumbMeterCurationDailyOperationalMeterStatusSid,
    NVL(ot.dumbMeterCurationDailyRecordDateSid, bt.dateSid) as dumbMeterCurationDailyRecordDateSid,
    NVL(ot.dumbMeterCurationDailyReceivedDateSid, -1) as dumbMeterCurationDailyReceivedDateSid,
    NVL(ot.dumbMeterCurationDailyReceivedDateTime, '1900-01-01 00:00:00') as dumbMeterCurationDailyReceivedDateTime,
    NVL(ot.dumbMeterCurationDailyCurationDateSid, -1) as dumbMeterCurationDailyCurationDateSid,
    NVL(ot.dumbMeterCurationDailyCurationDateTime, '1900-01-01 00:00:00') as dumbMeterCurationDailyCurationDateTime,
    NVL(ot.dumbMeterCurationDailyNoOfReadsInCuration, 0) as dumbMeterCurationDailyNoOfReadsInCuration,
    NVL(ot.dumbMeterCurationDailyHas11pmReadInd, 0) as dumbMeterCurationDailyHas11pmReadInd,
    NVL(ot.dumbMeterCurationDailyHasMidnightReadInd, 0) as dumbMeterCurationDailyHasMidnightReadInd,
    NVL(ot.dumbMeterCurationDailyCurationNumber, 0)  as dumbMeterCurationDailyCurationNumber

  FROM crossTable2 bt
  INNER JOIN otDailyReads ot on ot.dumbMeterCurationDailyAmrid = bt.dumbMeterAmrid AND ot.dumbMeterCurationDailyRecordDateSid = bt.dateSid 
  INNER JOIN meterStatus dsmms on dsmms.dumbMeterMeterStatusAmrid = bt.dumbMeterAmrid and dsmms.validFromSid <= bt.dateSid and dsmms.validToSid > bt.dateSid
  INNER JOIN operationalMeterStatus dsmoms on dsmoms.dumbMeterOperationalMeterStatusAmrid = bt.dumbMeterAmrid and dsmoms.validFromSid <= bt.dateSid and dsmoms.validToSid > bt.dateSid
  """)
  previousReads.createOrReplaceTempView("""previousReads""")
  
  # we join the two outputs for the full update
  baseJoined = spark.sql("""
  SELECT * from todayReads
  UNION
  SELECT * from previousReads
  """)
  

In [0]:
baseJoined.createOrReplaceTempView("""baseJoined""")

In [0]:
from ImaginaryCompany.ImaginaryWork.core import classImaginaryWork
import datetime
ImaginaryWorkVersion = int(classImaginaryWork().ImaginaryWorkVersion.split('.',1)[0])
print(ImaginaryWorkVersion)
modelEntityId = 2339
try:
  if ImaginaryWorkVersion >= 5:
    from ImaginaryCompany.ImaginaryWork.core import classTestLogAccess
    deltaDate = classTestLogAccess.getLatestSuccessfulStart(modelEntityId)
    if deltaDate == datetime.datetime(1900, 1, 1, 0, 0):
      deltaDate = None
  else:
    deltaDate = spark.sql(f"SELECT latestSuccessfulStart FROM ImaginaryCompany.entityLog WHERE entityId = {modelEntityId}").collect()[0][0]
except:
  deltaDate = None

if deltaDate is not None:
  sql = f"""dumbMeterCurationDailyDateCalendarYearWeek IN 
  (SELECT DISTINCT dumbMeterCurationDailyDateCalendarYearWeek FROM ImaginaryCompany_{env}.curated.facttest WHERE ImaginaryCompanyMetaDataLoadDateTime >= CAST('{deltaDate}' AS timestamp))"""
else:
  sql = "dumbMeterCurationDailyRecordDateSid >= date_format(current_date - INTERVAL 2 YEARS, 'yyyyMMdd')"
print(deltaDate)

In [0]:
views = [f"""
CREATE OR REPLACE VIEW ImaginaryCompany_{env}.views.viewfacttest AS
SELECT
  dumbMeterCurationDailyDateSid,
  dumbMeterCurationDailyDateCalendarYearWeek,
  dumbMeterCurationDailydumbMeterSid,
  dumbMeterCurationDailyUnInsSid,
  dumbMeterCurationDailyAddressSid,
  dumbMeterCurationDailyPremiseSid,
  dumbMeterCurationDailyMeterStatusSid,
  dumbMeterCurationDailyOperationalMeterStatusSid,
  dumbMeterCurationDailyRecordDateSid,
  dumbMeterCurationDailyReceivedDateSid,
  dumbMeterCurationDailyReceivedDateTime,
  dumbMeterCurationDailyCurationDateSid,
  dumbMeterCurationDailyCurationDateTime,
  dumbMeterCurationDailyNoOfReadsInCuration,
  dumbMeterCurationDailyHas11pmReadInd,
  dumbMeterCurationDailyHasMidnightReadInd,
  dumbMeterCurationDailyCurationNumber
FROM
  ImaginaryCompany_{env}.curated.facttest
WHERE
  {sql}
  """]

In [0]:
try:
  processedRows = fact.postProcess(baseJoined, views)
  batchEntity.end(entityId, processedRows)
  del batchEntity
except Exception as e:
  print(e)
  msg = str(e).replace("'", '').replace('"', '')  
  classTestLog.error(batchId, entityId, msg)

In [0]:
baseJoined.unpersist()

In [0]:
classTestEnTEx(batchId).triggerEntities(entityId)

In [0]:
dbutils.notebook.exit("facttest complete") #change to notebook name and delete this comment

# unitTest

In [0]:
#%sql
#select count(*) from baseJoined

In [0]:
%sql
--select dumbMeterCurationDailydumbMeterSid, dumbMeterCurationDailyRecordDateSid, dumbMeterCurationDailyReceivedDateSid, dumbMeterCurationDailyCurationDateSid, dumbMeterCurationDailyDateSid, count(*) from baseJoined
--group by dumbMeterCurationDailydumbMeterSid, dumbMeterCurationDailyRecordDateSid, dumbMeterCurationDailyReceivedDateSid, dumbMeterCurationDailyCurationDateSid, dumbMeterCurationDailyDateSid
--having count(*)>1
--duplicate check

In [0]:
%sql
--select dumbMeterCurationDailyNoOfReadsInCuration, count(*) from baseJoined
--group by dumbMeterCurationDailyNoOfReadsInCuration

In [0]:
#%sql
#select * from otdumbMeterReadsLatest
#where AMRID = 310524805
#and RecordTime > '2019-02-28'
#and RecordTime < '2019-02-29'


In [0]:
## only runs in DEV
## Provides some unit tests, particularly around the ImaginaryCompany config, percentage of nulls or default values in each column etc
## Change the name of the curated entity and the main source table (can do row count check)
#ft.testTemplate("factAppendTemplate","otMeterRead",baseJoined=baseJoined)

# source Table Metadata Test

If you need to check the metadata of any source tables, add here

In [0]:
## also only runs in DEV
## examples:
#ft.metadata(["sapImptt"], "ImaginaryCompany")
#ft.metadata(tables=["AUFK", "AFKO"], tableType="SAP") ## gets info from sapDd02vv, sapDd03vt
#ft.metadata(tables=["opfsexcelImaginaryCompanyordertypecategorydescription"], tableType="ImaginaryCompany")